## Imports y Carga de Modulo

In [4]:
# ==== Imports y carga del módulo de funciones (tuyas) ====
from pathlib import Path
import pandas as pd, numpy as np
import plotly.graph_objects as go
import importlib.util

# Cargar ../Analisis/Funciones_Drift.py como módulo
spec = importlib.util.spec_from_file_location("funciones_analisis", "../Analisis/Funciones_Drift.py")
funciones_analisis = importlib.util.module_from_spec(spec)
spec.loader.exec_module(funciones_analisis)

# Aliases a tus funciones
strip_outliers       = funciones_analisis.strip_outliers
compare_with_metric  = funciones_analisis.compare_with_metric

## Path y Carga de Modulo

In [5]:
# ==== Rutas ====
SERIES_PATH = Path("synthetic_data/synthetic_plant.csv")   # o "Series_Generadas/Series_Ajustadas.csv"
EVENTS_CSV  = Path("synthetic_data/etiquetado_manual.csv") # formato: date_time,variable,event

# ==== Parámetros base ====
RESAMPLE         = None          # o "15min", "1H"
RESAMPLE_AGG     = "median"
EXCLUDE_COLUMNS  = []
STRATEGIES       = ["decay","golden","seasonal"]
METRICS          = ["ks","mannwhitney","psi","wasserstein"]
CURRENT_WINDOWS  = ["6H","12H","1D"]   # <-- ventanas cortas para 35 días de serie

# ==== Helpers de tiempo ====
def _detect_time_col(df: pd.DataFrame) -> str:
    for c in ["date_time","datetime","timestamp","time","fecha","tiempo"]:
        if c in df.columns: return c
    c0 = df.columns[0]
    pd.to_datetime(df[c0], errors="raise")
    return c0

# ==== Cargar serie ====
assert SERIES_PATH.exists(), f"No encuentro {SERIES_PATH}"
raw = pd.read_csv(SERIES_PATH)
tcol = _detect_time_col(raw)
df = raw.rename(columns={tcol:"date_time"}).copy()
df["date_time"] = pd.to_datetime(df["date_time"], errors="coerce")
df = (df.dropna(subset=["date_time"])
        .sort_values("date_time")
        .set_index("date_time"))
df = strip_outliers(df)  # tu helper
num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
df = df[num_cols].copy()
assert len(num_cols) > 0, "No hay columnas numéricas."

# Límites de la serie
t_min, t_max = df.index.min(), df.index.max()
t_min, t_max

(Timestamp('2025-01-01 00:00:00'), Timestamp('2025-02-06 23:00:00'))

## Cargar Eventos Manuales

In [6]:
# ==== Cargar eventos manuales ====
assert EVENTS_CSV.exists(), f"No encuentro {EVENTS_CSV}"
events = pd.read_csv(EVENTS_CSV)
events["date_time"] = pd.to_datetime(events["date_time"], errors="coerce")
events = (events.dropna(subset=["date_time","variable","event"])
                .assign(event=lambda s: s["event"].str.lower().str.strip())
                .query("event in ['start','end']")
                .sort_values(["variable","date_time"])
                .reset_index(drop=True))

def events_to_intervals(ev: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    for var, g in ev.groupby("variable", sort=True):
        open_t=None
        for _,r in g.iterrows():
            if r["event"]=="start":
                open_t=r["date_time"]
            elif r["event"]=="end" and open_t is not None and r["date_time"]>open_t:
                rows.append({"column":var,"episode_start":open_t,"episode_end":r["date_time"]})
                open_t=None
    eps = pd.DataFrame(rows)
    if eps.empty:
        return eps
    # merge de solapes por variable
    out=[]
    for col, g in eps.groupby("column"):
        g = g.sort_values(["episode_start","episode_end"]).reset_index(drop=True)
        cs, ce = g.loc[0,"episode_start"], g.loc[0,"episode_end"]
        for _,r in g.iloc[1:].iterrows():
            s,e = r["episode_start"], r["episode_end"]
            if s <= ce:
                ce = max(ce, e)
            else:
                out.append({"column":col,"episode_start":cs,"episode_end":ce})
                cs, ce = s, e
        out.append({"column":col,"episode_start":cs,"episode_end":ce})
    return pd.DataFrame(out)

intervals = events_to_intervals(events)
intervals.head()

,column,episode_start,episode_end
0,var_1,2025-01-02 22:00:00,2025-01-03 05:00:00
1,var_1,2025-01-08 09:00:00,2025-01-09 22:00:00
2,var_1,2025-01-20 04:00:00,2025-01-22 20:00:00
3,var_1,2025-01-25 10:00:00,2025-01-25 15:00:00
4,var_1,2025-01-28 23:00:00,2025-02-06 22:00:00


In [7]:
# ==== Manual GT por ventana ====
def manual_drift_dict(intervals: pd.DataFrame, window_end: pd.Timestamp, window_len: str) -> dict:
    """Devuelve {var: True/False} si algún intervalo solapa [window_end - window_len, window_end]."""
    if intervals.empty:
        return {v: False for v in df.columns}
    start = window_end - pd.to_timedelta(window_len)
    out = {}
    for v in df.columns:
        g = intervals[intervals["column"]==v]
        out[v] = any((r.episode_end >= start) and (r.episode_start <= window_end) for _,r in g.iterrows())
    return out

## Runner

In [ ]:
def run_eval_for_window(window_len: str,
                        strategies=STRATEGIES, metrics=METRICS,
                        resample=RESAMPLE, resample_agg=RESAMPLE_AGG,
                        exclude=EXCLUDE_COLUMNS):
    """
    Ejecuta compare_with_metric para cada (strategy, metric) en la ventana actual [t_max - window_len, t_max],
    compara contra GT manual por variable (booleana) y devuelve:
      - detailed_df: una fila por variable–estrategia–métrica
      - col_eval:    agregación por estrategia–métrica
    """
    manual_gt = manual_drift_dict(intervals, t_max, window_len)
    agg_rows, detailed_rows = [], []

    for strategy in strategies:
        for metric in metrics:
            # Llamada directa a tu compare_with_metric con CURRENT_WINDOW=window_len
            dfm, overall = compare_with_metric(
                df=df.reset_index(),
                strategy=strategy,
                CURRENT_WINDOW=window_len,
                RESAMPLE=resample,
                RESAMPLE_AGG=resample_agg,
                EXCLUDE_COLUMNS=exclude,
                metric=metric,
                num_threshold=None,
            )
            # Fallback ligero para seasonal si vino vacío
            if (dfm is None or dfm.empty) and strategy=="seasonal":
                try:
                    dfm2, overall2 = compare_with_metric(
                        df=df.reset_index(),
                        strategy="seasonal",
                        CURRENT_WINDOW=window_len,
                        RESAMPLE=("1H" if resample is None else resample),
                        RESAMPLE_AGG=resample_agg,
                        EXCLUDE_COLUMNS=exclude,
                        metric=metric,
                        num_threshold=None,
                    )
                    if dfm2 is not None and not dfm2.empty:
                        dfm, overall = dfm2, overall2
                        print(f"[seasonal:{metric}] recuperado con RESAMPLE='1H' (window={window_len})")
                except TypeError:
                    pass

            if dfm is None or dfm.empty:
                print(f"[warn] Sin filas para {strategy}-{metric} (window={window_len}).")
                continue

            dfm = dfm.copy()
            dfm["variable"]     = dfm["col"].astype(str)
            dfm["auto_drift"]   = dfm["drift_detected"].fillna(False).astype(bool)
            dfm["manual_drift"] = dfm["variable"].map(manual_gt).fillna(False).astype(bool)

            # Detalle por variable
            dfm["TP"] = (dfm["auto_drift"] &  dfm["manual_drift"]).astype(int)
            dfm["FP"] = (dfm["auto_drift"] & ~dfm["manual_drift"]).astype(int)
            dfm["TN"] = (~dfm["auto_drift"] & ~dfm["manual_drift"]).astype(int)
            dfm["FN"] = (~dfm["auto_drift"] &  dfm["manual_drift"]).astype(int)
            dfm["strategy"] = strategy
            dfm["metric"]   = metric
            dfm["window"]   = window_len

            detailed_rows.append(
                dfm[["variable","strategy","metric","window","auto_drift","manual_drift","TP","FP","TN","FN"]]
            )

            # Agregado por estrategia–métrica
            agg = dfm[["TP","FP","TN","FN"]].sum().to_dict()
            TP,FP,TN,FN = agg.get("TP",0), agg.get("FP",0), agg.get("TN",0), agg.get("FN",0)
            tot = TP+FP+TN+FN
            acc = (TP+TN)/tot if tot else np.nan
            prec= TP/(TP+FP) if (TP+FP)>0 else np.nan
            rec = TP/(TP+FN) if (TP+FN)>0 else np.nan
            f1  = (2*prec*rec)/(prec+rec) if (prec and rec and (prec+rec)>0) else np.nan
            fpr = FP/(FP+TN) if (FP+TN)>0 else np.nan

            agg_rows.append({
                "window": window_len, "strategy": strategy, "metric": metric,
                "columns_eval": int(len(dfm)),
                "TP": TP, "FP": FP, "TN": TN, "FN": FN,
                "Accuracy": acc, "Precision": prec, "Recall": rec, "F1": f1, "FPR": fpr
            })

    detailed_df = (pd.concat(detailed_rows, ignore_index=True)
                     .sort_values(["window","variable","strategy","metric"]))
    col_eval    = (pd.DataFrame(agg_rows)
                     .sort_values(["window","strategy","metric"])
                     .reset_index(drop=True))

    print(f"[{window_len}] Esperado ≈ {len(df.columns)} × {len(STRATEGIES)} × {len(METRICS)} = {len(df.columns)*len(STRATEGIES)*len(METRICS)} filas")
    print(f"[{window_len}] detailed_df rows =", len(detailed_df))
    return detailed_df, col_eval

In [13]:
all_detailed = []
all_summary  = []

for win in CURRENT_WINDOWS:
    det, summ = run_eval_for_window(win)
    all_detailed.append(det)
    all_summary.append(summ)

detailed_df = pd.concat(all_detailed, ignore_index=True)
col_eval    = pd.concat(all_summary,  ignore_index=True)

display(col_eval)       # tabla agregada por (window, strategy, metric)
display(detailed_df)  # filas (variable, strategy, metric, window)

[6H] Esperado ≈ 10 × 3 × 4 = 120 filas
[6H] detailed_df rows = 120
[12H] Esperado ≈ 10 × 3 × 4 = 120 filas
[12H] detailed_df rows = 120
[1D] Esperado ≈ 10 × 3 × 4 = 120 filas
[1D] detailed_df rows = 120


,window,strategy,metric,columns_eval,TP,FP,TN,FN,Accuracy,Precision,Recall,F1,FPR
0,6H,decay,ks,10,7,3,0,0,0.7,0.700000,1.000000,0.823529,1.000000
1,6H,decay,mannwhitney,10,0,1,2,7,0.2,0.000000,0.000000,NaN,0.333333
2,6H,decay,psi,10,7,3,0,0,0.7,0.700000,1.000000,0.823529,1.000000
3,6H,decay,wasserstein,10,6,2,1,1,0.7,0.750000,0.857143,0.800000,0.666667
4,6H,golden,ks,10,7,3,0,0,0.7,0.700000,1.000000,0.823529,1.000000
5,6H,golden,mannwhitney,10,0,1,2,7,0.2,0.000000,0.000000,NaN,0.333333
6,6H,golden,psi,10,7,3,0,0,0.7,0.700000,1.000000,0.823529,1.000000
7,6H,golden,wasserstein,10,6,2,1,1,0.7,0.750000,0.857143,0.800000,0.666667
8,6H,seasonal,ks,10,7,3,0,0,0.7,0.700000,1.000000,0.823529,1.000000
9,6H,seasonal,mannwhitney,10,0,1,2,7,0.2,0.000000,0.000000,NaN,0.333333


,variable,strategy,metric,window,auto_drift,manual_drift,TP,FP,TN,FN
0,var_1,decay,ks,6H,True,True,1,0,0,0
1,var_1,decay,mannwhitney,6H,False,True,0,0,0,1
2,var_1,decay,psi,6H,True,True,1,0,0,0
3,var_1,decay,wasserstein,6H,True,True,1,0,0,0
4,var_1,golden,ks,6H,True,True,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...
355,var_9,golden,wasserstein,1D,True,True,1,0,0,0
356,var_9,seasonal,ks,1D,True,True,1,0,0,0
357,var_9,seasonal,mannwhitney,1D,False,True,0,0,0,1
358,var_9,seasonal,psi,1D,True,True,1,0,0,0


In [10]:
# Ver un mapa de aciertos por variable, por ventana
def hits_pivot(window="6H"):
    tmp = (detailed_df[detailed_df["window"]==window]
           .assign(hit=lambda x: np.select(
               [x["TP"]==1, x["FP"]==1, x["FN"]==1, x["TN"]==1],
               ["TP","FP","FN","TN"], default="NA")))
    pivot = tmp.pivot_table(index="variable",
                            columns=["strategy","metric"],
                            values="hit",
                            aggfunc="first")
    return pivot

display(hits_pivot("6H"))

strategy decay                             golden                              \
metric      ks mannwhitney psi wasserstein     ks mannwhitney psi wasserstein   
variable                                                                        
var_1       TP          FN  TP          TP     TP          FN  TP          TP   
var_10      TP          FN  TP          TP     TP          FN  TP          TP   
var_2       FP          TN  FP          FP     FP          TN  FP          FP   
var_3       TP          FN  TP          TP     TP          FN  TP          TP   
var_4       TP          FN  TP          FN     TP          FN  TP          FN   
var_5       FP          TN  FP          TN     FP          TN  FP          TN   
var_6       TP          FN  TP          TP     TP          FN  TP          TP   
var_7       TP          FN  TP          TP     TP          FN  TP          TP   
var_8       FP          FP  FP          FP     FP          FP  FP          FP   
var_9       TP          FN  TP          TP     TP          FN  TP          TP   

strategy seasonal                              
metric         ks mannwhitney psi wasserstein  
variable                                       
var_1          TP          FN  TP          TP  
var_10         TP          FN  TP          TP  
var_2          FP          TN  FP          FP  
var_3          TP          FN  TP          TP  
var_4          TP          FN  TP          TP  
var_5          FP          TN  FP          TN  
var_6          TP          FN  TP          TP  
var_7          TP          FN  TP          TP  
var_8          FP          FP  FP          FP  
var_9          TP          FN  TP          TP

In [12]:
def plot_window(var: str, strategy="golden", metric="psi", window="6H"):
    s = df[var]
    win_end = t_max
    win_start = win_end - pd.to_timedelta(window)

    # ¿Detectó auto?
    auto_on = False
    row = detailed_df[(detailed_df["variable"]==var) &
                      (detailed_df["strategy"]==strategy) &
                      (detailed_df["metric"]==metric) &
                      (detailed_df["window"]==window)]
    if not row.empty:
        auto_on = bool(row["auto_drift"].iloc[0])

    fig = go.Figure()
    fig.add_scatter(x=s.index, y=s.values, mode="lines", name=var)

    # Bandas manuales
    for _,r in intervals[intervals["column"]==var].iterrows():
        x0 = max(r["episode_start"], df.index.min())
        x1 = min(r["episode_end"],   df.index.max())
        if x1 > x0:
            fig.add_shape(type="rect", xref="x", yref="paper",
                          x0=x0, x1=x1, y0=0, y1=1,
                          fillcolor="rgba(200,60,60,0.20)", line=dict(color="rgba(200,60,60,0.8)"),
                          layer="below")

    # Banda auto (azul) en la ventana actual si detectó
    if auto_on:
        fig.add_shape(type="rect", xref="x", yref="paper",
                      x0=win_start, x1=win_end, y0=0, y1=1,
                      fillcolor="rgba(60,120,200,0.18)", line=dict(color="rgba(60,120,200,0.6)"),
                      layer="below")

    fig.update_layout(
        title=f"{var} — Manual (rojo) vs Auto {metric} [{strategy}] en {window}",
        xaxis=dict(title="Tiempo", rangeslider=dict(visible=True)),
        yaxis=dict(title="Valor")
    )
    fig.show()

# Ejemplos:
plot_window("var_1", strategy="golden", metric="psi", window="6H")
plot_window("var_10", strategy="decay",  metric="ks",  window="12H")


In [14]:
# === Promedios y rankings a partir de col_eval ===
import numpy as np
import pandas as pd

score_cols = ["Accuracy","Precision","Recall","F1","FPR"]

# 1) Promedio por tamaño de ventana (macro-promedio simple)
avg_by_window = (col_eval
                 .groupby("window")[score_cols]
                 .mean()
                 .sort_index())
display(avg_by_window.style.format("{:.3f}").set_caption("Promedio por ventana (macro)"))

# 1b) Promedio por ventana ponderado por columns_eval (macro ponderado)
def _wavg(group):
    w = group["columns_eval"].clip(lower=1).values
    out = {}
    for c in score_cols:
        vals = group[c].astype(float).values
        ok = np.isfinite(vals)
        out[c] = np.average(vals[ok], weights=w[ok]) if ok.any() else np.nan
    return pd.Series(out)

wavg_by_window = col_eval.groupby("window").apply(_wavg)
display(wavg_by_window.style.format("{:.3f}").set_caption("Promedio por ventana (ponderado)"))

# 2) Promedio por combinación (estrategia, métrica), agregando sobre ventanas
avg_by_combo = (col_eval
                .groupby(["strategy","metric"])[score_cols]
                .mean()
                .sort_values("F1", ascending=False))
display(avg_by_combo.style.format("{:.3f}").set_caption("Promedio por combinación (promedio sobre ventanas)"))

# 3) Promedio por (ventana, estrategia) y por (ventana, métrica) para ver perfiles
avg_by_window_strategy = (col_eval
                          .groupby(["window","strategy"])[score_cols]
                          .mean()
                          .sort_values(["window","F1"], ascending=[True, False]))
display(avg_by_window_strategy.style.format("{:.3f}").set_caption("Promedio por (ventana, estrategia)"))

avg_by_window_metric = (col_eval
                        .groupby(["window","metric"])[score_cols]
                        .mean()
                        .sort_values(["window","F1"], ascending=[True, False]))
display(avg_by_window_metric.style.format("{:.3f}").set_caption("Promedio por (ventana, métrica)"))

# 4) Tabla “3×4×3”: una fila por (window, strategy, metric)
avg_full = (col_eval
            .groupby(["window","strategy","metric"])[score_cols]
            .mean()
            .reset_index()
            .sort_values(["window","F1"], ascending=[True, False]))
display(avg_full.style.format("{:.3f}").set_caption("Ventana × Estrategia × Métrica (promedios)"))

# 5) Ranking top-10 combinaciones por F1 promedio (sobre todas las ventanas)
top10 = avg_by_combo.reset_index().sort_values("F1", ascending=False).head(10)
display(top10.style.format("{:.3f}").set_caption("Top-10 combinaciones por F1 (promedio sobre ventanas)"))


,Accuracy,Precision,Recall,F1,FPR
window,,,,,
12H,0.608,0.566,0.726,0.843,0.667
1D,0.608,0.608,0.726,0.835,0.667
6H,0.583,0.540,0.726,0.824,0.750


,Accuracy,Precision,Recall,F1,FPR
window,,,,,
12H,0.608,0.566,0.726,0.843,0.667
1D,0.608,0.608,0.726,0.835,0.667
6H,0.583,0.540,0.726,0.824,0.750


ValueError: Unknown format code 'f' for object of type 'str'

ValueError: Unknown format code 'f' for object of type 'str'

In [18]:
import plotly.graph_objects as go
import plotly.express as px

def plot_detections(
    var: str,
    strategies=None,
    metrics=None,
    show_manual=False,
    window_filter=None,
):
    """
    Muestra sólo las detecciones automáticas de drift para una variable.

    Parámetros:
      var: nombre de la variable (columna)
      strategies: lista opcional de estrategias (e.g., ["golden", "decay"])
      metrics: lista opcional de métricas (e.g., ["psi", "ks"])
      show_manual: si True, también muestra los intervalos manuales (rojo)
      window_filter: si no es None, filtra sólo esa ventana (e.g. "6H" o "1D")
    """
    if var not in df.columns:
        raise ValueError(f"{var} no está en df.columns")

    data = df[var]
    fig = go.Figure()
    fig.add_scatter(x=data.index, y=data.values, mode="lines", name=var)

    # --- Filtro del detailed_df ---
    sel = detailed_df[detailed_df["variable"] == var].copy()
    if strategies:
        sel = sel[sel["strategy"].isin(strategies)]
    if metrics:
        sel = sel[sel["metric"].isin(metrics)]
    if window_filter:
        sel = sel[sel["window"] == window_filter]
    if sel.empty:
        print("⚠️ No hay detecciones automáticas para ese filtro.")
        return

    # --- Color base por estrategia ---
    palette = px.colors.qualitative.Set2
    strat_colors = {s: palette[i % len(palette)] for i, s in enumerate(STRATEGIES)}

    # --- Bandas de detección ---
    offset_minutes = 5  # desplazar ligeramente cada métrica
    for i, (_, row) in enumerate(sel.iterrows()):
        if not row["auto_drift"]:
            continue
        sname = row["strategy"]
        mname = row["metric"]
        win = row["window"]

        # Calcula la ventana temporal
        win_end = df.index.max()
        win_len = pd.to_timedelta(win)
        win_start = win_end - win_len

        # Offset vertical para distinguir métricas
        y0 = -0.02 - i * 0.02
        color = strat_colors.get(sname, "gray")

        fig.add_shape(
            type="rect", xref="x", yref="paper",
            x0=win_start, x1=win_end, y0=y0, y1=y0 + 0.015,
            fillcolor=color, opacity=0.45,
            line=dict(width=0),
            layer="above",
        )

        fig.add_annotation(
            x=win_start, y=y0 + 0.007, xref="x", yref="paper",
            text=f"{sname}/{mname}", showarrow=False,
            font=dict(size=10, color="black")
        )

    # --- Opcional: intervalos manuales ---
    if show_manual and not intervals.empty:
        for _, r in intervals[intervals["column"] == var].iterrows():
            x0, x1 = r["episode_start"], r["episode_end"]
            fig.add_shape(
                type="rect", xref="x", yref="paper",
                x0=x0, x1=x1, y0=0, y1=1,
                fillcolor="rgba(200,60,60,0.15)",
                line=dict(color="rgba(200,60,60,0.8)", width=1),
                layer="below"
            )

    fig.update_layout(
        title=f"{var} — Detecciones automáticas por método",
        xaxis=dict(title="Tiempo", rangeslider=dict(visible=True)),
        yaxis=dict(title="Valor"),
        showlegend=False,
        height=400,
        margin=dict(t=50, b=40)
    )
    fig.show()


In [19]:
# Sólo detecciones (sin bandas manuales)
plot_detections("var_1")

# Mostrar sólo estrategias “golden” y “seasonal”
plot_detections("var_1", strategies=["golden", "seasonal"])

# Mostrar sólo métricas PSI y Wasserstein
plot_detections("var_10", metrics=["psi", "wasserstein"])

# Mostrar detecciones de una sola ventana (ej. 12H)
plot_detections("var_3", window_filter="12H")

# Comparar auto + manual
plot_detections("var_5", show_manual=True)


In [20]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def _win_limits(window: str):
    win_end = df.index.max()
    win_start = win_end - pd.to_timedelta(window)
    return win_start, win_end

def _auto_detected(var: str, strategy: str, metric: str, window: str) -> bool:
    sel = detailed_df[(detailed_df["variable"]==var) &
                      (detailed_df["strategy"]==strategy) &
                      (detailed_df["metric"]==metric) &
                      (detailed_df["window"]==window)]
    return (not sel.empty) and bool(sel["auto_drift"].iloc[0])

In [21]:
def plot_grid_by_combo(strategy: str, metric: str, window: str = "6H", show_manual: bool = False):
    vars10 = list(df.columns)[:10]  # ajusta si quieres otro orden/subset
    r, c = 2, 5
    win_start, win_end = _win_limits(window)

    fig = make_subplots(rows=r, cols=c,
                        subplot_titles=[f"{v}" for v in vars10],
                        horizontal_spacing=0.06, vertical_spacing=0.18)

    for i, var in enumerate(vars10, start=1):
        row = 1 if i <= 5 else 2
        col = i if i <= 5 else i-5

        s = df[var]
        fig.add_trace(go.Scatter(x=s.index, y=s.values, mode="lines", name=var,
                                 line=dict(width=1)), row=row, col=col)

        # Manual (opcional)
        if show_manual and not intervals.empty:
            for _, rct in intervals[intervals["column"]==var].iterrows():
                x0 = max(rct["episode_start"], df.index.min())
                x1 = min(rct["episode_end"],   df.index.max())
                if x1 > x0:
                    fig.add_shape(type="rect", xref=f"x{(row-1)*5+col}", yref=f"y{(row-1)*5+col}",
                                  x0=x0, x1=x1, y0=min(s), y1=max(s),
                                  fillcolor="rgba(200,60,60,0.15)", line=dict(width=0))

        # Auto: sombrear ventana solo si detectó
        if _auto_detected(var, strategy, metric, window):
            fig.add_shape(type="rect", xref=f"x{(row-1)*5+col}", yref=f"y{(row-1)*5+col}",
                          x0=win_start, x1=win_end, y0=min(s), y1=max(s),
                          fillcolor="rgba(60,120,200,0.28)", line=dict(color="rgba(60,120,200,0.9)", width=1))

        # Ejes
        fig.update_xaxes(showgrid=True, row=row, col=col)
        fig.update_yaxes(showgrid=True, row=row, col=col)

    fig.update_layout(height=650, width=1200, showlegend=False,
                      title=f"{strategy} / {metric} — ventana {window} (azul = detección)")
    fig.show()


In [28]:
# Cambia la ventana según necesites: "6H", "12H", "1D", etc.
WIN = "1D"

# PSI (3 plots: decay/golden/seasonal)
plot_grid_by_combo("decay",   "psi", WIN, show_manual=False)
plot_grid_by_combo("golden",  "psi", WIN, show_manual=False)
plot_grid_by_combo("seasonal","psi", WIN, show_manual=False)

# WASSERSTEIN
plot_grid_by_combo("decay",   "wasserstein", WIN, show_manual=False)
plot_grid_by_combo("golden",  "wasserstein", WIN, show_manual=False)
plot_grid_by_combo("seasonal","wasserstein", WIN, show_manual=False)

# KS
plot_grid_by_combo("decay",   "ks", WIN, show_manual=False)
plot_grid_by_combo("golden",  "ks", WIN, show_manual=False)
plot_grid_by_combo("seasonal","ks", WIN, show_manual=False)

# MANN–WHITNEY
plot_grid_by_combo("decay",   "mannwhitney", WIN, show_manual=False)
plot_grid_by_combo("golden",  "mannwhitney", WIN, show_manual=False)
plot_grid_by_combo("seasonal","mannwhitney", WIN, show_manual=False)


In [29]:
import pandas as pd
from tqdm import tqdm

def sweep_detections_over_time(
    window="12H",
    step="6H",
    strategies=("decay","golden","seasonal"),
    metrics=("ks","mannwhitney","psi","wasserstein"),
    exclude_columns=None,
    resample=None,
    resample_agg="median",
):
    """
    Recorre la serie en ventanas deslizantes. Para cada t_end:
      - usa df_slice = df.loc[:t_end]
      - llama a compare_with_metric(df_slice.reset_index(), ...)
      - registra detecciones por variable como intervalos [t_end - window, t_end]
    Devuelve un DataFrame con columnas:
      ['variable','strategy','metric','t0','t1','window']
    """
    if exclude_columns is None: exclude_columns = []
    w = pd.to_timedelta(window)
    starts_at = df.index.min() + w
    ends_at   = df.index.max()
    t_ends = pd.date_range(starts_at, ends_at, freq=step, inclusive="both")

    out = []
    for t_end in tqdm(t_ends, desc=f"sweep {window} step {step}"):
        t0 = t_end - w
        if t0 < df.index.min():  # seguridad
            continue
        df_slice = df.loc[:t_end]

        for strategy in strategies:
            for metric in metrics:
                dfm, _ = compare_with_metric(
                    df=df_slice.reset_index(),       # tu función espera 'date_time' en columnas
                    strategy=strategy,
                    CURRENT_WINDOW=window,
                    RESAMPLE=resample,
                    RESAMPLE_AGG=resample_agg,
                    EXCLUDE_COLUMNS=exclude_columns,
                    metric=metric,
                    num_threshold=None,
                )
                if dfm is None or dfm.empty:
                    continue
                # variable + flag
                dfm = dfm.rename(columns={"col":"variable"})
                for _, r in dfm.iterrows():
                    if bool(r.get("drift_detected", False)):
                        out.append({
                            "variable": str(r["variable"]),
                            "strategy": strategy,
                            "metric": metric,
                            "t0": t0,
                            "t1": t_end,
                            "window": window
                        })
    return pd.DataFrame(out)

# === EJECUTA UNA VEZ (ajusta ventana/step a tu gusto) ===
timeline_df = sweep_detections_over_time(
    window="12H",   # prueba también "6H" o "1D"
    step="6H",      # granularidad del barrido
)
display(timeline_df.head())


sweep 12H step 6H: 100%|██████████| 146/146 [05:44<00:00,  2.36s/it]


,variable,strategy,metric,t0,t1,window
0,var_1,decay,ks,2025-01-01 06:00:00,2025-01-01 18:00:00,12H
1,var_10,decay,ks,2025-01-01 06:00:00,2025-01-01 18:00:00,12H
2,var_2,decay,ks,2025-01-01 06:00:00,2025-01-01 18:00:00,12H
3,var_3,decay,ks,2025-01-01 06:00:00,2025-01-01 18:00:00,12H
4,var_4,decay,ks,2025-01-01 06:00:00,2025-01-01 18:00:00,12H


In [34]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_grid_timeline(strategy: str, metric: str, window: str, show_manual=False):
    """
    Dibuja 10 variables (2×5). Para cada una:
      - serie en azul
      - TODAS las ventanas [t0,t1] donde ese (strategy, metric, window) detectó (bandas azules).
      - opcional: bandas manuales en rojo.
    """
    assert "timeline_df" in globals(), "Primero corre sweep_detections_over_time()"
    subset = timeline_df[(timeline_df["strategy"]==strategy) &
                         (timeline_df["metric"]==metric) &
                         (timeline_df["window"]==window)]
    if subset.empty:
        print("No hay detecciones para esa combinación.")
        return

    vars10 = list(df.columns)[:10]
    fig = make_subplots(rows=2, cols=5,
                        subplot_titles=vars10,
                        horizontal_spacing=0.06, vertical_spacing=0.18)

    for i, var in enumerate(vars10, start=1):
        row = 1 if i <= 5 else 2
        col = i if i <= 5 else i-5

        s = df[var]
        fig.add_trace(go.Scatter(x=s.index, y=s.values, mode="lines", name=var, line=dict(width=1)),
                      row=row, col=col)

        # manual (opcional)
        if show_manual and 'intervals' in globals() and not intervals.empty:
            for _, r in intervals[intervals["column"]==var].iterrows():
                x0, x1 = r["episode_start"], r["episode_end"]
                fig.add_shape(type="rect", xref=f"x{(row-1)*5+col}", yref=f"y{(row-1)*5+col}",
                              x0=x0, x1=x1, y0=min(s), y1=max(s),
                              fillcolor="rgba(200,60,60,0.15)", line=dict(width=0))

        # TODAS las detecciones auto de este método para esta variable
        hits = subset[subset["variable"]==var]
        for _, h in hits.iterrows():
            fig.add_shape(type="rect", xref=f"x{(row-1)*5+col}", yref=f"y{(row-1)*5+col}",
                          x0=h["t0"], x1=h["t1"], y0=min(s), y1=max(s),
                          fillcolor="rgba(60,120,200,0.28)",
                          line=dict(color="rgba(60,120,200,0.9)", width=1))

        fig.update_xaxes(showgrid=True, row=row, col=col)
        fig.update_yaxes(showgrid=True, row=row, col=col)

    fig.update_layout(height=650, width=1200, showlegend=False,
                      title=f"{strategy} / {metric} — timeline de detecciones (window={window}, step en sweep)")
    fig.show()

# === EJEMPLOS ===
# plot_grid_timeline("golden","psi","12H", show_manual=True)
plot_grid_timeline("decay","wasserstein","12H", show_manual=False)
# plot_grid_timeline("decay","ks","6H", show_manual=True)
